<a href="https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My baseline rule

I will prioritize content items that have meaningful search demand but weaker average search position.

The baseline score will combine GSC impressions as a volume signal and GSC average position as an opportunity signal.

Higher impressions increase the opportunity score, while a worse average position increases the opportunity score within a reasonable ranking range.

### Reason codes

- `HIGH_VOLUME_LOW_POSITION` — high impressions with a weaker average position.
- `HIGH_VOLUME` — high impressions with a less concerning position.
- `LOW_VOLUME_OPPORTUNITY` — lower-volume content that still shows some improvement opportunity.

In [11]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

%pip -q install duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("DuckDB connected and March 2026 partition selected")

HF_TOKEN loaded: True
DuckDB connected and March 2026 partition selected


In [12]:
volume_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions < 10 THEN '1-9'
            WHEN gsc_impressions < 100 THEN '10-99'
            WHEN gsc_impressions < 1000 THEN '100-999'
            ELSE '1000+'
        END AS impressions_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM {REL}
    WHERE gsc_data_available = TRUE
    GROUP BY 1
    ORDER BY
        CASE impressions_bucket
            WHEN '0' THEN 1
            WHEN '1-9' THEN 2
            WHEN '10-99' THEN 3
            WHEN '100-999' THEN 4
            ELSE 5
        END
""").df()

volume_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_bucket,n,avg_clicks
0,1-9,1463532,0.01
1,10-99,1508921,0.10
2,100-999,606189,0.80
3,1000+,32419,5.07


Signal: GSC average position
Why I chose it: Average position is a search-performance signal that can help identify content with visibility but room for improvement. I will check whether impression and click levels differ across position buckets.
Verdict: To be determined from the observed bucket results.

In [13]:
position_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position = 0 THEN 'No data'
            WHEN gsc_avg_position <= 10 THEN '1-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            WHEN gsc_avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM {REL}
    WHERE gsc_data_available = TRUE
    GROUP BY 1
    ORDER BY
        CASE position_bucket
            WHEN 'No data' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-20' THEN 3
            WHEN '21-50' THEN 4
            ELSE 5
        END
""").df()

position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_impressions,avg_clicks
0,No data,163189,2.87,0.01
1,1-10,2020295,94.73,0.32
2,11-20,519223,56.60,0.18
3,21-50,631491,88.59,0.12
4,51+,276863,12.53,0.01


Verdict: CONFIRMED — The observed buckets show that pages in positions 1–10 have the highest average clicks, while pages at 51+ have very low average clicks. The relationship is not perfectly monotonic across every bucket, so position is treated as a directional signal rather than a standalone predictor.

### Baseline rule

I will rank content using a simple opportunity score based on search volume and average position. Higher impressions indicate more search visibility, while positions 11–20 provide an opportunity to improve ranking. The rule will assign one reason code and one action label to each content item.

The score is a decision-support baseline, not a prediction of future performance.

In [14]:
import pandas as pd
# Build the March 2026 baseline ranked queue

baseline = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Monthly search volume and clicks
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        -- Average observed search position
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position

    FROM {REL}

    WHERE gsc_data_available = TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# ---------------------------------------------------------
# 1. Calculate the opportunity score
# ---------------------------------------------------------

def calculate_score(row):

    impressions = row["impressions"]
    position = row["avg_position"]

    # Volume component
    if impressions >= 1000:
        volume_score = 50
    elif impressions >= 100:
        volume_score = 35
    elif impressions >= 10:
        volume_score = 20
    else:
        volume_score = 0

    # Position component
    if pd.isna(position):
        position_score = 0
    elif 11 <= position <= 20:
        position_score = 50
    elif 21 <= position <= 50:
        position_score = 35
    elif position <= 10:
        position_score = 15
    else:
        position_score = 5

    return volume_score + position_score


baseline["score"] = baseline.apply(calculate_score, axis=1)


# ---------------------------------------------------------
# 2. Assign ONE reason code
# ---------------------------------------------------------

def assign_reason(row):

    position = row["avg_position"]
    impressions = row["impressions"]

    if not pd.isna(position) and 11 <= position <= 20 and impressions >= 100:
        return "POSITION_OPPORTUNITY"

    elif impressions >= 1000:
        return "HIGH_VOLUME"

    elif not pd.isna(position) and 21 <= position <= 50 and impressions >= 100:
        return "RANKING_OPPORTUNITY"

    else:
        return "LOW_PRIORITY"


baseline["reason_code"] = baseline.apply(assign_reason, axis=1)


# ---------------------------------------------------------
# 3. Assign ONE action label
# ---------------------------------------------------------

def assign_action(score):

    if score >= 85:
        return "REFRESH"

    elif score >= 50:
        return "REVIEW"

    else:
        return "MONITOR"


baseline["action"] = baseline["score"].apply(assign_action)


# ---------------------------------------------------------
# 4. Rank everything
# ---------------------------------------------------------

baseline = baseline.sort_values(
    by=["score", "impressions", "clicks"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1


# ---------------------------------------------------------
# 5. Arrange final columns
# ---------------------------------------------------------

baseline = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]


# ---------------------------------------------------------
# 6. Show the top 20
# ---------------------------------------------------------

print("Total content items ranked:", len(baseline))

baseline.head(20)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total content items ranked: 176738


,rank,client_hash_id,content_hash_id,impressions,clicks,avg_position,score,reason_code,action
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,100,POSITION_OPPORTUNITY,REFRESH
1,2,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,782.0,18.615742,100,POSITION_OPPORTUNITY,REFRESH
2,3,client_23a62021009f63c4,content_5e1c049f62e33b11,120175.0,168.0,18.077081,100,POSITION_OPPORTUNITY,REFRESH
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,11.967474,100,POSITION_OPPORTUNITY,REFRESH
4,5,client_23a62021009f63c4,content_f6723f0229e1bfdc,69822.0,13.0,15.766691,100,POSITION_OPPORTUNITY,REFRESH
5,6,client_23a62021009f63c4,content_65c75874a23fca87,64935.0,18.0,13.571837,100,POSITION_OPPORTUNITY,REFRESH
6,7,client_23a62021009f63c4,content_2690f62f39fb14fe,61074.0,101.0,17.739450,100,POSITION_OPPORTUNITY,REFRESH
7,8,client_23a62021009f63c4,content_1df00c7789a0adcc,54156.0,134.0,18.235795,100,POSITION_OPPORTUNITY,REFRESH
8,9,client_23a62021009f63c4,content_5be6be2550a98fc5,53059.0,162.0,17.045527,100,POSITION_OPPORTUNITY,REFRESH
9,10,client_20259bd6705d81d4,content_653bbcddf2314227,51183.0,20.0,18.368758,100,POSITION_OPPORTUNITY,REFRESH


In [15]:
# Write the ranked queue to the required output path

import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows written:", len(baseline))

Saved: work/outputs/baseline_action_score.csv
Rows written: 176738


### Top-20 review

I reviewed the top 20 items produced by my baseline rule. For each item, I record the action, reason code, a confidence note, and what could make the recommendation wrong.

The review is decision-support only. The baseline uses observed March 2026 signals and does not use future-window or label-derived information.

In [16]:
import pandas as pd

# Load the ranked queue created in Section 2
ranked_queue = pd.read_csv("work/outputs/baseline_action_score.csv")

# Take the top 20 ranked items
top20 = ranked_queue.head(20).copy()

# Add a confidence note and a "what would make it wrong" note
top20["confidence_note"] = (
    "Moderate confidence: the item matches the baseline rule using observed signals."
)

top20["what_would_make_it_wrong"] = (
    "The recommendation could be wrong if the observed position/click pattern "
    "does not represent a real refresh opportunity or if the data is incomplete."
)

# Select the fields required for the review
top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("Top-20 items reviewed:", len(top20_review))

top20_review


Top-20 items reviewed: 20


,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
1,2,client_23a62021009f63c4,content_66288edeb93b7c4f,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
2,3,client_23a62021009f63c4,content_5e1c049f62e33b11,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
4,5,client_23a62021009f63c4,content_f6723f0229e1bfdc,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
5,6,client_23a62021009f63c4,content_65c75874a23fca87,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
6,7,client_23a62021009f63c4,content_2690f62f39fb14fe,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
7,8,client_23a62021009f63c4,content_1df00c7789a0adcc,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
8,9,client_23a62021009f63c4,content_5be6be2550a98fc5,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...
9,10,client_20259bd6705d81d4,content_653bbcddf2314227,REFRESH,POSITION_OPPORTUNITY,Moderate confidence: the item matches the base...,The recommendation could be wrong if the obser...


### Weak picks + leakage check

I reviewed the baseline's highest-ranked items for possible weak picks. A high score does not guarantee that a refresh will improve performance, so the recommendations are treated as decision-support rather than guaranteed outcomes.

The baseline uses only observed March 2026 performance signals. No future-window outcome or label-derived field is used in the scoring rule.

In [17]:
# ---------------------------------------------------------
# 1. Find potentially weak picks in the top 20
# ---------------------------------------------------------

weak_picks = top20[
    (top20["impressions"] < 100) |
    (top20["avg_position"] > 50) |
    (top20["avg_position"].isna())
].copy()

print("Potential weak picks in top 20:", len(weak_picks))

weak_picks[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]


Potential weak picks in top 20: 0


,rank,client_hash_id,content_hash_id,impressions,avg_position,score,reason_code,action


In [18]:
# ---------------------------------------------------------
# 2. Leakage check
# ---------------------------------------------------------

allowed_features = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
}

excluded_from_rule = {
    "trend_pct",
    "trend_direction",
    "is_declining_label"
}

print("Allowed baseline signals:")
print(sorted(allowed_features))

print("\nExcluded label/future-derived fields:")
print(sorted(excluded_from_rule))

print("\nBaseline rule uses only observed March 2026 signals.")
print("No future-window or label-derived fields are used.")

Allowed baseline signals:
['gsc_avg_position', 'gsc_clicks', 'gsc_impressions']

Excluded label/future-derived fields:
['is_declining_label', 'trend_direction', 'trend_pct']

Baseline rule uses only observed March 2026 signals.
No future-window or label-derived fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.